# Notebook 0: Quickstart & Project Overview

Start here. This notebook gives an overview of the full pipeline and provides a simple test to confirm your environment is working.

---

## Project Purpose
Automatically detect and flag sensitive content in a large archive of TIFF images using two methods:
- **Method 1 — Semantic Similarity:** Generate image descriptions via LLaVA, then match against a sensitivity taxonomy using sentence embeddings.
- **Method 2 — LLaVA Direct Classification:** Prompt LLaVA directly with a structured moderation prompt for each image.

---

## Notebook Order

| # | Notebook | Purpose |
|---|----------|---------|
| 0 | `00_Quickstart.ipynb` *(this file)* | Environment check, file indexing |
| 1 | `01_Downsampling_Images.ipynb` | Resize TIFFs + extract metadata |
| 2 | `02_Metadata_Extraction.ipynb` | Deep EXIF/IPTC metadata extraction |
| 3 | `03_Sensitivity_Flagging_LLaVA.ipynb` | Method 2: LLaVA direct classification |
| 4 | `04_Sensitivity_Flagging_Semantic.ipynb` | Method 1: Semantic similarity classification |

---

## Sensitive Content Categories

The project monitors for 9 categories (defined in `data/Sensitive_Content_Taxonomy_DataFrame.csv`):

1. Historical racialized performance
2. Human remains
3. Native American imagery
4. Nudity/sexual content
5. Violence/graphic content
6. Hate symbols
7. Medical/health records
8. Student/PII records
9. Other sensitive categories

---

## Prerequisites

Install Python dependencies:
```bash
pip install pillow pandas tifffile sentence-transformers torch tqdm iptcinfo3
```

Install and set up Ollama with LLaVA:
```bash
# Install Ollama from https://ollama.com
ollama pull llava-llama3
```

## Step 1: Index Your Image Files

In [ ]:
import os
import pandas as pd

# ============================================================
# Set the path to your images folder
# ============================================================
IMAGE_DIR = r"path/to/your/images"  # e.g. r"C:\Downloads\Numbered Scans_1"

# Supported image extensions
SUPPORTED_EXT = ('.tif', '.tiff', '.jpg', '.jpeg', '.png')

file_data = []
for root, _, files in os.walk(IMAGE_DIR):
    for file in files:
        if file.lower().endswith(SUPPORTED_EXT):
            full_path = os.path.join(root, file)
            file_data.append({
                'file_path': full_path,
                'file_name': file,
                'file_id': os.path.splitext(file)[0],
                'extension': os.path.splitext(file)[1].lstrip('.'),
            })

df_files = pd.DataFrame(file_data)
print(f"Found {len(df_files)} image files.")
df_files.head(10)

## Step 2: Quick Environment Check

In [ ]:
# Check that all required libraries are available
checks = {}

try:
    import PIL; checks['Pillow'] = f'OK (v{PIL.__version__})'
except ImportError:
    checks['Pillow'] = 'MISSING — pip install pillow'

try:
    import tifffile; checks['tifffile'] = 'OK'
except ImportError:
    checks['tifffile'] = 'MISSING — pip install tifffile'

try:
    import sentence_transformers; checks['sentence-transformers'] = 'OK'
except ImportError:
    checks['sentence-transformers'] = 'MISSING — pip install sentence-transformers'

try:
    import torch; checks['torch'] = f'OK (v{torch.__version__})'
except ImportError:
    checks['torch'] = 'MISSING — pip install torch'

try:
    import iptcinfo3; checks['iptcinfo3'] = 'OK'
except ImportError:
    checks['iptcinfo3'] = 'OPTIONAL — pip install iptcinfo3'

print("Environment check:")
for lib, status in checks.items():
    icon = '✅' if 'OK' in status else ('⚠️' if 'OPTIONAL' in status else '❌')
    print(f"  {icon} {lib}: {status}")

## Step 3: Test Ollama Connection

In [ ]:
import subprocess
import platform

# Set path to Ollama
if platform.system() == "Windows":
    OLLAMA_PATH = r"C:\Users\YOUR_USERNAME\AppData\Local\Programs\Ollama\ollama.exe"
else:
    OLLAMA_PATH = "/usr/local/bin/ollama"

# List available models
try:
    result = subprocess.run([OLLAMA_PATH, "list"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("✅ Ollama is running. Available models:")
        print(result.stdout)
    else:
        print(f"❌ Ollama error: {result.stderr}")
except FileNotFoundError:
    print(f"❌ Ollama not found at: {OLLAMA_PATH}")
    print("   Update OLLAMA_PATH above, or install Ollama from https://ollama.com")
except subprocess.TimeoutExpired:
    print("❌ Ollama timed out — make sure it's running.")

## Step 4: Load the Sensitive Content Taxonomy

In [ ]:
import pandas as pd

# Adjust path if running from a different location
TAXONOMY_PATH = r"../data/Sensitive_Content_Taxonomy_DataFrame.csv"

try:
    taxonomy_df = pd.read_csv(TAXONOMY_PATH)
    print(f"✅ Taxonomy loaded: {len(taxonomy_df)} categories")
    taxonomy_df[["Label", "Summary"]]
except FileNotFoundError:
    print(f"❌ Taxonomy file not found at: {TAXONOMY_PATH}")
    print("   Check that the data/ folder is in the right place.")

---

## Next Steps

Once your environment is set up, proceed to the notebooks in order:

1. **`01_Downsampling_Images.ipynb`** — convert and resize your TIFF images
2. **`02_Metadata_Extraction.ipynb`** — extract EXIF/IPTC metadata
3. **`03_Sensitivity_Flagging_LLaVA.ipynb`** — run LLaVA direct classification
4. **`04_Sensitivity_Flagging_Semantic.ipynb`** — run semantic similarity classification